### attendance -> certificate eligibility (APSSDC ML internship 2026)

40 sessions total, present if you stayed 90+ min, need 32/40 (80%) present to get the certificate.
Don't have the real zoom export yet so simulating data below to get the pipeline working, will swap in real CSV later.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

TOTAL_SESSIONS = 40
SESSION_DURATION = 120     # mins
PRESENT_THRESHOLD = 90
CERT_THRESHOLD = 0.80
NUM_STUDENTS = 1000
SEED = 42

np.random.seed(SEED)


In [ ]:
# fake data generator - buckets students into behaviour types so it's
# not just random noise. numbers for the ranges are just guesses based
# on what seemed reasonable, not from any real distribution

def make_students(n=NUM_STUDENTS):
    rng = np.random.default_rng(SEED)
    rows = []

    for i in range(n):
        roll = rng.random()

        if roll < 0.35:
            kind = "consistent"      # 32-40 sessions
        elif roll < 0.65:
            kind = "moderate"        # 24-31
        elif roll < 0.82:
            kind = "at_risk"         # 12-23
        else:
            kind = "irregular"       # 0-11

        if kind == "consistent":
            present = int(rng.integers(32, 41))
        elif kind == "moderate":
            present = int(rng.integers(24, 32))
        elif kind == "at_risk":
            present = int(rng.integers(12, 24))
        else:
            present = int(rng.integers(0, 12))

        present = min(present, TOTAL_SESSIONS)
        absent = TOTAL_SESSIONS - present

        if present > 0:
            if kind == "consistent":
                avg_dur = round(float(rng.uniform(100, 120)), 1)
            elif kind == "moderate":
                avg_dur = round(float(rng.uniform(90, 110)), 1)
            elif kind == "at_risk":
                avg_dur = round(float(rng.uniform(90, 100)), 1)
            else:
                avg_dur = round(float(rng.uniform(90, 95)), 1)
        else:
            avg_dur = 0.0

        total_mins = int(present * avg_dur)
        early_dropouts = int(rng.integers(0, max(1, absent // 2 + 1)))
        late_joins = int(rng.integers(0, max(1, absent // 3 + 1)))

        pct = round((present / TOTAL_SESSIONS) * 100, 2)
        eligible = 1 if pct >= 80 else 0

        rows.append({
            "student_id": f"APSSDC{1001+i:04d}",
            "student_type": kind,
            "sessions_present": present,
            "sessions_absent": absent,
            "avg_duration_min": avg_dur,
            "total_mins_attended": total_mins,
            "early_dropouts": early_dropouts,
            "late_joins": late_joins,
            "attendance_pct": pct,
            "certificate_eligible": eligible,
        })

    return pd.DataFrame(rows)

df = make_students()
print(df.shape)
df.head()


In [ ]:
# checking the split isn't totally lopsided before going further
df['certificate_eligible'].value_counts(normalize=True)


In [ ]:
# extra ratio-based features on top of the raw counts - these tend to
# generalize better than raw numbers since attendance % matters more
# than absolute session count. engagement_score is just something I
# put together, weights are a guess and could probably use tuning

def add_features(data):
    data = data.copy()
    data['dropout_rate'] = data['early_dropouts'] / data['sessions_present'].replace(0, 1)
    data['late_join_rate'] = data['late_joins'] / data['sessions_present'].replace(0, 1)
    data['attendance_ratio'] = data['sessions_present'] / TOTAL_SESSIONS
    data['duration_efficiency'] = data['avg_duration_min'] / SESSION_DURATION

    data['engagement_score'] = (
        data['attendance_ratio'] * 0.55
        + data['duration_efficiency'] * 0.25
        - data['dropout_rate'] * 0.12
        - data['late_join_rate'] * 0.08
    ).clip(0, 1)

    return data

df = add_features(df)

FEATURES = [
    'sessions_present', 'avg_duration_min', 'total_mins_attended',
    'early_dropouts', 'late_joins', 'dropout_rate', 'late_join_rate',
    'attendance_ratio', 'duration_efficiency', 'engagement_score'
]
TARGET = 'certificate_eligible'

df[FEATURES].describe()


In [ ]:
X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(len(X_train), 'train /', len(X_test), 'test')


In [ ]:
# random forest - no scaling needed, handles the mix of raw counts and
# ratio features fine, and gives feature importance for the report

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=5,
    max_features='sqrt',
    class_weight='balanced',
    random_state=SEED,
    n_jobs=-1,
)

rf.fit(X_train, y_train)


In [ ]:
# first tried this with default params (n_estimators=100, no max_depth)
# and it just memorised the training set, so added max_depth + min_samples_leaf
# rf_v1 = RandomForestClassifier(random_state=SEED).fit(X_train, y_train)
# rf_v1.score(X_train, y_train)  # was 1.0, that's how I noticed the overfit


In [ ]:
preds = rf.predict(X_test)
probs = rf.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, preds)
auc = roc_auc_score(y_test, probs)
cm = confusion_matrix(y_test, preds)

print('accuracy:', round(acc * 100, 2), '%')
print('roc auc: ', round(auc, 4))
print()
print(cm)
print()
print(classification_report(y_test, preds, target_names=['Not Eligible', 'Eligible']))


In [ ]:
importances = pd.DataFrame({
    'feature': FEATURES,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

importances


In [ ]:
# also want to see this holds up outside just the one train/test split
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_scores = cross_val_score(rf, X, y, cv=skf, scoring='accuracy')

for i, s in enumerate(cv_scores, 1):
    print(f'fold {i}: {s*100:.2f}%')
print('mean:', round(cv_scores.mean() * 100, 2), '%')


In [ ]:
# quick helper for a single student - plug in their numbers, get a
# prediction back. useful for one-off "will I get the cert" questions

def predict_student(sessions_present, avg_duration, early_dropouts, late_joins):
    sessions_absent = TOTAL_SESSIONS - sessions_present
    total_mins = int(sessions_present * avg_duration)
    dropout_rate = early_dropouts / max(sessions_present, 1)
    late_join_rate = late_joins / max(sessions_present, 1)
    attendance_ratio = sessions_present / TOTAL_SESSIONS
    duration_efficiency = avg_duration / SESSION_DURATION
    engagement_score = max(0, min(1,
        attendance_ratio * 0.55
        + duration_efficiency * 0.25
        - dropout_rate * 0.12
        - late_join_rate * 0.08
    ))

    row = pd.DataFrame([{
        'sessions_present': sessions_present,
        'avg_duration_min': avg_duration,
        'total_mins_attended': total_mins,
        'early_dropouts': early_dropouts,
        'late_joins': late_joins,
        'dropout_rate': dropout_rate,
        'late_join_rate': late_join_rate,
        'attendance_ratio': attendance_ratio,
        'duration_efficiency': duration_efficiency,
        'engagement_score': engagement_score,
    }])

    pred = rf.predict(row)[0]
    prob = rf.predict_proba(row)[0]

    return {
        'sessions_present': sessions_present,
        'attendance_pct': round(attendance_ratio * 100, 1),
        'prediction': 'Eligible' if pred == 1 else 'Not Eligible',
        'prob_eligible': round(prob[1] * 100, 1),
    }

# sanity check on a few made up cases before trusting it
test_cases = [
    (38, 112, 1, 0),
    (32, 95, 3, 2),
    (28, 100, 4, 3),
    (15, 92, 8, 5),
    (5, 45, 10, 8),
]

for sessions, dur, drop, late in test_cases:
    print(predict_student(sessions, dur, drop, late))


In [ ]:
# 32/95/3/2 came back Eligible with only ~57% confidence which makes sense,
# that one's right on the 80% boundary so the model isn't super sure either way


In [ ]:
# batch predictions over everyone + save to csv, coordinators wanted this
# to cross check against the plain 80% rule

all_preds = rf.predict(df[FEATURES])
all_probs = rf.predict_proba(df[FEATURES])[:, 1]

df['ml_prediction'] = ['Eligible' if p == 1 else 'Not Eligible' for p in all_preds]
df['ml_prob_eligible'] = (all_probs * 100).round(1)

df.to_csv('attendance_predictions.csv', index=False)
df.head()


In [ ]:
att = df['attendance_pct']
print('avg attendance:', round(att.mean(), 1), '%')
print('min/max:', att.min(), '/', att.max())

eligible = df['certificate_eligible'].sum()
at_risk = ((att >= 60) & (att < 80)).sum()
not_eligible = (att < 60).sum()

print('eligible:', eligible, f'({eligible/10:.1f}%)')
print('at risk (60-79%):', at_risk, f'({at_risk/10:.1f}%)')
print('not eligible:', not_eligible, f'({not_eligible/10:.1f}%)')

# ml predictions line up with the rule-based cutoff pretty closely, good enough
# for now - could probably drop total_mins_attended since it's basically a
# copy of sessions_present * avg_duration, didn't get around to checking that
